### Bibliotecas

In [1]:
import pandas as pd
import numpy as np

import warnings
import os

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import RandomizedSearchCV, KFold, StratifiedKFold
from sklearn.metrics import roc_auc_score
from scipy.stats import randint, uniform
import numpy as np

In [2]:
# Deinir diretório
os.chdir('G:/Meu Drive/MeuDrive2/academico/3.kaggle/8.predicting_loan_payback')

# Exibir linhas e colunas
pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)

# Evitar avisos
warnings.filterwarnings('ignore')

### EDA

Importando os dados

In [3]:
# treino
train = pd.read_csv('train.csv')

# teste
test = pd.read_csv('test.csv')

# arquivo de submissão
sample_submission = pd.read_csv('sample_submission.csv')

In [4]:
# Remover a coluna id, ela é desnecessária nos dados de treino
train = train.drop(columns=["id"])

In [5]:
# Primeiras observações
train.head()

,annual_income,debt_to_income_ratio,credit_score,loan_amount,interest_rate,gender,marital_status,education_level,employment_status,loan_purpose,grade_subgrade,loan_paid_back
0,29367.99,0.084,736,2528.42,13.67,Female,Single,High School,Self-employed,Other,C3,1.0
1,22108.02,0.166,636,4593.10,12.92,Male,Married,Master's,Employed,Debt consolidation,D3,0.0
2,49566.20,0.097,694,17005.15,9.76,Male,Single,High School,Employed,Debt consolidation,C5,1.0
3,46858.25,0.065,533,4682.48,16.10,Female,Single,High School,Employed,Debt consolidation,F1,1.0
4,25496.70,0.053,665,12184.43,10.21,Male,Married,High School,Employed,Other,D1,1.0


In [6]:
# Verifica se há dados faltantes
train.isna().sum()

annual_income           0
debt_to_income_ratio    0
credit_score            0
loan_amount             0
interest_rate           0
gender                  0
marital_status          0
education_level         0
employment_status       0
loan_purpose            0
grade_subgrade          0
loan_paid_back          0
dtype: int64

In [7]:
# Formato do conjunto de treino
train.shape

(593994, 12)

In [8]:
# Descrição do conjunto
train.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 593994 entries, 0 to 593993
Data columns (total 12 columns):
 #   Column                Non-Null Count   Dtype  
---  ------                --------------   -----  
 0   annual_income         593994 non-null  float64
 1   debt_to_income_ratio  593994 non-null  float64
 2   credit_score          593994 non-null  int64  
 3   loan_amount           593994 non-null  float64
 4   interest_rate         593994 non-null  float64
 5   gender                593994 non-null  object 
 6   marital_status        593994 non-null  object 
 7   education_level       593994 non-null  object 
 8   employment_status     593994 non-null  object 
 9   loan_purpose          593994 non-null  object 
 10  grade_subgrade        593994 non-null  object 
 11  loan_paid_back        593994 non-null  float64
dtypes: float64(5), int64(1), object(6)
memory usage: 54.4+ MB


In [9]:
# Número de valores únique por coluna do tipo object
for col in train.select_dtypes(include='object').columns:
    print(col, train[col].nunique())

gender 3
marital_status 4
education_level 5
employment_status 5
loan_purpose 8
grade_subgrade 30


In [10]:
"""# Estilo geral
sns.set_theme(style="whitegrid", palette="crest", font_scale=1.2)

# Colunas categóricas
cat_cols = ['gender', 'marital_status', 'education_level', 'employment_status', 'loan_purpose', 'grade_subgrade']

# Criar figura 2x3
fig, axes = plt.subplots(2, 3, figsize=(22, 12))
axes = axes.flatten()

# Plotar cada coluna categórica
for i, col in enumerate(cat_cols):
    ax = axes[i]
    plot = sns.barplot(
        x=col,
        y='loan_paid_back',
        data=train,
        estimator=lambda x: x.mean(),
        ax=ax,
        ci=None
    )

    # Adicionar rótulos de valor acima das barras
    for p in plot.patches:
        height = p.get_height()
        ax.annotate(f'{height:.2f}', 
                    (p.get_x() + p.get_width() / 2., height),
                    ha='center', va='bottom', fontsize=10, color='black', xytext=(0, 3),
                    textcoords='offset points')

    # Personalização dos eixos
    ax.set_title(f'Média de Loan Paid Back por {col}', fontsize=14, fontweight='bold')
    ax.set_xlabel('')
    ax.set_ylabel('Média de Loan Paid Back', fontsize=12)
    ax.tick_params(axis='x', rotation=40)
    ax.set_ylim(0, 1)  # média entre 0 e 1
    sns.despine(ax=ax, left=True, bottom=True)

# Desligar subplots extras se houver
for j in range(len(cat_cols), len(axes)):
    axes[j].axis('off')

# Título geral
fig.suptitle('Média de Pagamento de Empréstimo por Categoria', fontsize=18, fontweight='bold', y=1.02)

plt.tight_layout()
plt.show()"""

'# Estilo geral\nsns.set_theme(style="whitegrid", palette="crest", font_scale=1.2)\n\n# Colunas categóricas\ncat_cols = [\'gender\', \'marital_status\', \'education_level\', \'employment_status\', \'loan_purpose\', \'grade_subgrade\']\n\n# Criar figura 2x3\nfig, axes = plt.subplots(2, 3, figsize=(22, 12))\naxes = axes.flatten()\n\n# Plotar cada coluna categórica\nfor i, col in enumerate(cat_cols):\n    ax = axes[i]\n    plot = sns.barplot(\n        x=col,\n        y=\'loan_paid_back\',\n        data=train,\n        estimator=lambda x: x.mean(),\n        ax=ax,\n        ci=None\n    )\n\n    # Adicionar rótulos de valor acima das barras\n    for p in plot.patches:\n        height = p.get_height()\n        ax.annotate(f\'{height:.2f}\', \n                    (p.get_x() + p.get_width() / 2., height),\n                    ha=\'center\', va=\'bottom\', fontsize=10, color=\'black\', xytext=(0, 3),\n                    textcoords=\'offset points\')\n\n    # Personalização dos eixos\n    ax.se

Verificando se há muitos outliers nas colunas numéricas

In [11]:
"""# Estilo geral
sns.set_theme(style="whitegrid", palette="crest")

# Selecionar apenas colunas numéricas (float + int), desconsiderando 'loan_paid_back'
num_cols = train.select_dtypes(include=['float', 'int']).columns.drop(['loan_paid_back'])

# Definir número de linhas e colunas automaticamente (layout próximo de quadrado)
n_cols = 2  # número de colunas por linha
n_rows = int(np.ceil(len(num_cols) / n_cols))

fig, axes = plt.subplots(n_rows, n_cols, figsize=(n_cols * 6, n_rows * 5))
axes = axes.flatten()

# Loop para plotar cada coluna
for i, col in enumerate(num_cols):
    sns.boxplot(y=train[col], ax=axes[i])
    axes[i].set_title(f'Boxplot de {col}', fontsize=12, fontweight='bold')

# Desligar eixos extras, caso existam
for j in range(len(num_cols), len(axes)):
    axes[j].axis('off')

plt.tight_layout()
plt.show()"""

'# Estilo geral\nsns.set_theme(style="whitegrid", palette="crest")\n\n# Selecionar apenas colunas numéricas (float + int), desconsiderando \'loan_paid_back\'\nnum_cols = train.select_dtypes(include=[\'float\', \'int\']).columns.drop([\'loan_paid_back\'])\n\n# Definir número de linhas e colunas automaticamente (layout próximo de quadrado)\nn_cols = 2  # número de colunas por linha\nn_rows = int(np.ceil(len(num_cols) / n_cols))\n\nfig, axes = plt.subplots(n_rows, n_cols, figsize=(n_cols * 6, n_rows * 5))\naxes = axes.flatten()\n\n# Loop para plotar cada coluna\nfor i, col in enumerate(num_cols):\n    sns.boxplot(y=train[col], ax=axes[i])\n    axes[i].set_title(f\'Boxplot de {col}\', fontsize=12, fontweight=\'bold\')\n\n# Desligar eixos extras, caso existam\nfor j in range(len(num_cols), len(axes)):\n    axes[j].axis(\'off\')\n\nplt.tight_layout()\nplt.show()'

### Separando em treino e validação

In [9]:
# Separando os dados
from sklearn.model_selection import train_test_split

target = "loan_paid_back"
X = train.drop(columns=[target])
y = train[target]

X_train, X_valid, y_train, y_valid = train_test_split(X, y, 
                                                      test_size=0.2, 
                                                      random_state=42, 
                                                      shuffle=True)

### Codificando

In [10]:
# Dummies
from sklearn.preprocessing import OneHotEncoder

# Colunas categóricas para dummies
cat_cols = ['gender', 'marital_status', 'education_level', 'employment_status', 'loan_purpose']

# Criar encoder
encoder = OneHotEncoder(drop='first', sparse_output=False, handle_unknown='ignore')

# Aprender dummies no X_train
X_train_encoded = pd.DataFrame(
    encoder.fit_transform(X_train[cat_cols]),
    columns=encoder.get_feature_names_out(cat_cols),
    index=X_train.index
)

# Remover as colunas originais categóricas e juntar dummies
X_train = X_train.drop(columns=cat_cols).join(X_train_encoded)

# Aplicar o mesmo encoder em X_valid
X_valid_encoded = pd.DataFrame(
    encoder.transform(X_valid[cat_cols]),
    columns=encoder.get_feature_names_out(cat_cols),
    index=X_valid.index
)

X_valid = X_valid.drop(columns=cat_cols).join(X_valid_encoded)

In [11]:
# OrdinalEncoder
from sklearn.preprocessing import OrdinalEncoder

# Coluna a codificar
col_ord = ['grade_subgrade']

# Criar encoder ordinal
ord_encoder = OrdinalEncoder()

# Treino
X_train[col_ord] = ord_encoder.fit_transform(X_train[col_ord])

# Validação (aplica o mesmo mapeamento)
X_valid[col_ord] = ord_encoder.transform(X_valid[col_ord])

### Outliers

In [12]:
num_cols = ['annual_income', 'debt_to_income_ratio', 'credit_score', 'loan_amount', 'interest_rate']

# Criar cópias para não sobrescrever original
X_train_out = X_train.copy()
X_valid_out = X_valid.copy()

# Percorrer apenas as colunas que você quer
for col in num_cols:
    lower = X_train_out[col].quantile(0.01)
    upper = X_train_out[col].quantile(0.99)
    
    # Winsorizing no treino
    X_train_out[col] = X_train_out[col].clip(lower, upper)
    
    # Aplicar MESMOS limites no valid
    X_valid_out[col] = X_valid_out[col].clip(lower, upper)

### Padronização

In [13]:
from sklearn.preprocessing import MinMaxScaler

# Criar o scaler
scaler = MinMaxScaler()

# Ajustar apenas no treino e transformar
X_train_scaled = scaler.fit_transform(X_train_out)

# Aplicar a mesma transformação no conjunto de validação
X_valid_scaled = scaler.transform(X_valid_out)

# Transformar de volta em DataFrame mantendo os nomes das colunas
X_train_scaled = pd.DataFrame(X_train_scaled, columns=X_train_out.columns)
X_valid_scaled = pd.DataFrame(X_valid_scaled, columns=X_valid_out.columns)

### Previsão

##### XGBoost

In [20]:
from xgboost import XGBClassifier
from sklearn.model_selection import RandomizedSearchCV, StratifiedKFold
from sklearn.metrics import roc_auc_score
import numpy as np

# X_train_scaled, X_valid_scaled, y_train, y_valid JÁ PRONTOS

# 1. Calcular scale_pos_weight (útil pro desbalanceamento 20/80)
n_pos = np.sum(y_train == 1)
n_neg = np.sum(y_train == 0)
scale_pos_weight = n_neg / n_pos
print("scale_pos_weight:", scale_pos_weight)

# 2. Definir modelo base do XGBoost
xgb_model = XGBClassifier(
    random_state=42,
    n_estimators=200,         # valor base (será sobrescrito na busca)
    n_jobs=-1,
    objective="binary:logistic",
    eval_metric="auc",
    use_label_encoder=False,
    scale_pos_weight=scale_pos_weight
)

# 3. Espaço de busca de hiperparâmetros
param_dist_xgb = {
    "n_estimators": [100, 200, 300, 500],
    "max_depth": [3, 4, 5, 8],
    "learning_rate": [0.01, 0.05, 0.1, 0.2],
    "subsample": [0.6, 0.8, 1.0],
    "colsample_bytree": [0.6, 0.8, 1.0],
    "min_child_weight": [1, 3, 5],
    "gamma": [0, 0.1, 0.2],
}

# 4. Validação cruzada
cv = StratifiedKFold(n_splits=5, 
                     shuffle=True, 
                     random_state=42)

# 5. RandomizedSearchCV com XGBoost
random_search_xgb = RandomizedSearchCV(
    estimator=xgb_model,
    param_distributions=param_dist_xgb,
    n_iter=30,
    scoring="roc_auc",
    n_jobs=2,
    cv=cv,
    verbose=3,
    random_state=42,
)

# 6. Treinar busca aleatória
random_search_xgb.fit(X_train_scaled, y_train)

print("Best XGBoost CV AUC:", random_search_xgb.best_score_)
print("Best XGBoost params:", random_search_xgb.best_params_)

# 7. Melhor modelo e avaliação em validação
best_xgb = random_search_xgb.best_estimator_

y_valid_proba_xgb = best_xgb.predict_proba(X_valid_scaled)[:, 1]
auc_valid_xgb = roc_auc_score(y_valid, y_valid_proba_xgb)
print("Valid AUC (XGBoost):", auc_valid_xgb)

# Fitting 5 folds for each of 30 candidates, totalling 150 fits
# Best XGBoost CV AUC: 0.9210836758852208
# Best XGBoost params: {'subsample': 1.0, 'n_estimators': 500, 'min_child_weight': 1, 'max_depth': 3, 'learning_rate': 0.2, 'gamma': 0.1, 'colsample_bytree': 0.8}
# Valid AUC (XGBoost): 0.9223628354028068

scale_pos_weight: 0.25152755391211823
Fitting 5 folds for each of 30 candidates, totalling 150 fits
Best XGBoost CV AUC: 0.9210836758852208
Best XGBoost params: {'subsample': 1.0, 'n_estimators': 500, 'min_child_weight': 1, 'max_depth': 3, 'learning_rate': 0.2, 'gamma': 0.1, 'colsample_bytree': 0.8}
Valid AUC (XGBoost): 0.9223628354028068


##### DecisionTree

In [29]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import RandomizedSearchCV, StratifiedKFold
from sklearn.metrics import roc_auc_score
import numpy as np

# X_train_scaled, X_valid_scaled, y_train, y_valid JÁ PRONTOS

# 1. Modelo base da Decision Tree
dt_model = DecisionTreeClassifier(
    random_state=42,
    class_weight="balanced"   # ajusta pesos das classes para lidar com 20/80
)

# 2. Espaço de busca de hiperparâmetros
param_dist_dt = {
    "max_depth": [None, 3, 4, 5, 8, 10, 15],
    "min_samples_split": [2, 5, 10, 20],
    "min_samples_leaf": [1, 2, 4, 8],
    "max_features": [None, "sqrt", "log2"],
    "criterion": ["gini", "entropy", "log_loss"],
}

# 3. Validação cruzada estratificada (mesmo esquema de antes)
cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

# 4. RandomizedSearchCV para Decision Tree
random_search_dt = RandomizedSearchCV(
    estimator=dt_model,
    param_distributions=param_dist_dt,
    n_iter=30,              # mesmo número de iterações
    scoring="roc_auc",      # mesma métrica
    n_jobs=2,
    cv=cv,
    verbose=3,
    random_state=42,
)

# 5. Treinar a busca aleatória
random_search_dt.fit(X_train_scaled, y_train)

print("Best Decision Tree CV AUC:", random_search_dt.best_score_)
print("Best Decision Tree params:", random_search_dt.best_params_)

# 6. Melhor modelo e avaliação em validação
best_dt = random_search_dt.best_estimator_

y_valid_proba_dt = best_dt.predict_proba(X_valid_scaled)[:, 1]
auc_valid_dt = roc_auc_score(y_valid, y_valid_proba_dt)
print("Valid AUC (Decision Tree):", auc_valid_dt)

# Fitting 5 folds for each of 30 candidates, totalling 150 fits
# Best Decision Tree CV AUC: 0.9118137262767675
# Best Decision Tree params: {'min_samples_split': 20, 'min_samples_leaf': 2, 'max_features': None, 'max_depth': 10, 'criterion': 'gini'}
# Valid AUC (Decision Tree): 0.9130768670226026

Fitting 5 folds for each of 30 candidates, totalling 150 fits
Best Decision Tree CV AUC: 0.9118137262767675
Best Decision Tree params: {'min_samples_split': 20, 'min_samples_leaf': 2, 'max_features': None, 'max_depth': 10, 'criterion': 'gini'}
Valid AUC (Decision Tree): 0.9130768670226026


##### Modelo LightGBM

In [31]:
from lightgbm import LGBMClassifier
from sklearn.model_selection import RandomizedSearchCV, StratifiedKFold
from sklearn.metrics import roc_auc_score
import numpy as np

# X_train_scaled, X_valid_scaled, y_train, y_valid JÁ PRONTOS

# 1. Calcular scale_pos_weight (útil pro desbalanceamento 20/80)
n_pos = np.sum(y_train == 1)
n_neg = np.sum(y_train == 0)
scale_pos_weight = n_neg / n_pos
print("scale_pos_weight:", scale_pos_weight)

# 2. Modelo base LightGBM
lgbm_model = LGBMClassifier(
    random_state=42,
    n_estimators=200,          # valor base (vai ser ajustado na busca)
    objective="binary",
    n_jobs=-1,
    class_weight=None,         # vamos usar scale_pos_weight
    scale_pos_weight=scale_pos_weight,
    boosting_type="gbdt",
    metric="auc"
)

# 3. Espaço de busca de hiperparâmetros
param_dist_lgbm = {
    "n_estimators": [100, 200, 300, 500],
    "max_depth": [-1, 3, 5, 8, 10],          # -1 = sem limite
    "learning_rate": [0.01, 0.05, 0.1, 0.2],
    "num_leaves": [15, 31, 63, 127],
    "subsample": [0.6, 0.8, 1.0],            # bagging_fraction
    "colsample_bytree": [0.6, 0.8, 1.0],     # feature_fraction
    "min_child_samples": [5, 10, 20, 30],
    "reg_alpha": [0, 0.1, 0.5, 1.0],         # L1
    "reg_lambda": [0, 0.1, 0.5, 1.0],        # L2
}

# 4. Validação cruzada estratificada (mesmo padrão)
cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

# 5. RandomizedSearchCV com LightGBM
random_search_lgbm = RandomizedSearchCV(
    estimator=lgbm_model,
    param_distributions=param_dist_lgbm,
    n_iter=30,
    scoring="roc_auc",
    n_jobs=2,
    cv=cv,
    verbose=3,
    random_state=42,
)

# 6. Treinar a busca aleatória
random_search_lgbm.fit(X_train_scaled, y_train)

print("Best LightGBM CV AUC:", random_search_lgbm.best_score_)
print("Best LightGBM params:", random_search_lgbm.best_params_)

# 7. Melhor modelo e avaliação em validação
best_lgbm = random_search_lgbm.best_estimator_

y_valid_proba_lgbm = best_lgbm.predict_proba(X_valid_scaled)[:, 1]
auc_valid_lgbm = roc_auc_score(y_valid, y_valid_proba_lgbm)
print("Valid AUC (LightGBM):", auc_valid_lgbm)

# Best LightGBM CV AUC: 0.922814861426934
# Best LightGBM params: {'subsample': 1.0, 'reg_lambda': 1.0, 'reg_alpha': 0.1, 'num_leaves': 15, 'n_estimators': 500, 'min_child_samples': 20, 'max_depth': 3, 'learning_rate': 0.2, 'colsample_bytree': 0.8}
# Valid AUC (LightGBM): 0.9241184903111026

scale_pos_weight: 0.25152755391211823
Fitting 5 folds for each of 30 candidates, totalling 150 fits
[LightGBM] [Warning] Found whitespace in feature_names, replace with underlines
[LightGBM] [Info] Number of positive: 379692, number of negative: 95503
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.073723 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 1324
[LightGBM] [Info] Number of data points in the train set: 475195, number of used features: 26
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.799024 -> initscore=1.380203
[LightGBM] [Info] Start training from score 1.380203
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No fu

In [32]:
print("Valid AUC XGBoost     :", auc_valid_xgb)
print("Valid AUC LightGBM    :", auc_valid_lgbm)
print("Valid AUC DecisionTree:", auc_valid_dt)

Valid AUC XGBoost     : 0.9223628354028068
Valid AUC LightGBM    : 0.9241184903111026
Valid AUC DecisionTree: 0.9130768670226026


### Dados de teste

In [33]:
# Removendo a coluna id do X_test (mas vamos precisar dela para o submission)
ids = test["id"].copy()
X_test = test.drop(columns=["id"])

##### Codificando

In [34]:
# Transformar X_test usando o encoder já treinado no treino
X_test_encoded = pd.DataFrame(
    encoder.transform(X_test[cat_cols]),
    columns=encoder.get_feature_names_out(cat_cols),
    index=X_test.index
)

# Remover as colunas originais categóricas e juntar as dummies
X_test = X_test.drop(columns=cat_cols).join(X_test_encoded)

In [35]:
# Aplicar a codificação ordinal no X_test
X_test[col_ord] = ord_encoder.transform(X_test[col_ord])

##### Outliers

In [36]:
# Criar cópia do X_test
X_test_out = X_test.copy()

# Aplicar winsorizing usando os limites do treino
for col in num_cols:
    lower = X_train_out[col].quantile(0.01)
    upper = X_train_out[col].quantile(0.99)
    
    X_test_out[col] = X_test_out[col].clip(lower, upper)

##### Padronização

In [37]:
# Aplicar a mesma transformação no X_test
X_test_scaled = scaler.transform(X_test_out)

# Transformar de volta em DataFrame
X_test_scaled = pd.DataFrame(X_test_scaled, columns=X_test_out.columns, index=X_test_out.index)

##### Previsão

In [38]:
# Prever no conjunto de teste
y_test_pred = best_lgbm.predict(X_test_scaled)

### Submissão

In [39]:
# Criar DataFrame de submissão
submission = pd.DataFrame({
    "id": ids,         # ids do conjunto de teste
    "y": y_test_pred   # substituir "y" pelo nome da target da competição, se necessário
})

# Salvar em CSV
submission.to_csv("best_lgbm.csv", index=False)